CacheBackedEmbeddings

영구적으로 임베딩을 지정하는 LocalFileStore

In [ ]:
# [목적] LocalFileStore를 이용해 임베딩 결과를 디스크에 저장하는 캐시 예제를 준비합니다.
# OpenAI 임베딩 모델과 파일 기반 저장소를 만든 뒤, 이후 같은 문서를 다시 처리할 때 결과를 재사용합니다.
# API 호출 시간과 비용을 줄이고, 프로그램을 다시 실행해도 캐시를 유지하기 위해 필요합니다.
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()

logging.langsmith("Chapter11-Embeddings")

# OpenAIEmbeddings는 텍스트의 의미를 비교할 수 있는 숫자 벡터로 변환하는 LangChain 객체입니다.
embedding = OpenAIEmbeddings()

# LocalFileStore는 캐시 데이터를 ./cache/ 폴더의 파일로 보관하는 저장소입니다.
store = LocalFileStore("./cache/")

In [ ]:
# [목적] 파일 저장소를 사용하는 캐시형 임베딩 객체를 생성합니다.
# CacheBackedEmbeddings는 먼저 저장된 벡터를 찾고, 없을 때만 OpenAI 임베딩 모델을 호출합니다.
# namespace에 모델 이름을 넣어 서로 다른 모델의 캐시가 섞이지 않도록 합니다.
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding,
    document_embedding_cache=store,
    namespace=embedding.model,
)

In [ ]:
# [목적] 파일 기반 캐시에 현재 저장된 임베딩 항목을 확인합니다.
# yield_keys는 저장소의 키를 차례로 반환하며, list로 변환해 한 번에 표시합니다.
# 아직 캐시가 비어 있는지 또는 이전 실행 결과가 남아 있는지 점검하는 데 사용합니다.
list(store.yield_keys())

In [ ]:
# [목적] 텍스트 파일을 읽어 벡터 검색에 사용할 문서 조각으로 나눕니다.
# TextLoader로 파일을 불러온 뒤 CharacterTextSplitter가 최대 1,000자 단위로 문서를 분할합니다.
# 긴 문서를 적절한 크기로 나눠야 각 조각의 의미를 임베딩하고 검색 결과로 돌려줄 수 있습니다.
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader("./data/appendix-keywords.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)

In [ ]:
# [목적] 문서 조각을 임베딩해 FAISS 벡터 검색 데이터베이스를 처음 생성합니다.
# FAISS.from_documents가 각 문서를 cached_embedder로 변환하고, %time이 실행 시간을 측정합니다.
# 최초 실행에서는 캐시가 비어 있어 필요한 임베딩을 생성한 후 검색용 db에 저장합니다.
%time db = FAISS.from_documents(documents, cached_embedder)

In [ ]:
# [목적] 같은 문서로 벡터 데이터베이스를 다시 만들어 캐시 효과를 확인합니다.
# 앞 셀에서 저장된 임베딩이 있으면 cached_embedder가 이를 재사용해 새 API 호출을 줄입니다.
# %time으로 첫 실행과의 시간을 비교하면 파일 기반 임베딩 캐시의 이점을 알 수 있습니다.
%time db2 = FAISS.from_documents(documents, cached_embedder)

비영구적으로 임베딩을 저장하는 InMemoryByteStore

In [ ]:
# [목적] 실행 중인 메모리에만 임베딩 캐시를 저장하는 예제를 설정합니다.
# InMemoryByteStore를 CacheBackedEmbeddings에 연결해 같은 실행 안에서는 임베딩 결과를 재사용합니다.
# 프로그램이 종료되면 캐시는 사라지므로, 영구 보관이 필요 없는 짧은 작업에 적합합니다.
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import InMemoryByteStore

# InMemoryByteStore는 파일 대신 현재 파이썬 실행 메모리에 키와 값을 보관합니다.
store = InMemoryByteStore()

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding,
    store,
    namespace=embedding.model
)